# 1. Check GPU

In [ ]:
#@title 1. Check GPU
import subprocess, sys
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit(
        "NO GPU ATTACHED.\n"
        "Runtime > Change runtime type > Hardware accelerator > T4 GPU, then rerun.\n"
        "Without this the model loads onto CPU and each request takes minutes."
    )
print(out.stdout)


# 2. Mount Drive and cache weights there

In [ ]:
#@title 2. Mount Drive and cache weights there
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# A fresh Colab session otherwise re-downloads ~7GB of weights, which is
# several minutes of dead air before the demo can start.
HF_HOME = "/content/drive/MyDrive/hf-cache"
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

cached = list(Path(HF_HOME).glob("hub/models--Qwen*"))
print(f"HF_HOME = {HF_HOME}")
print("CACHE HIT - weights already on Drive" if cached else "CACHE MISS - first run will download ~7GB")


# 3. Clone the repo and install

In [ ]:
#@title 3. Clone the repo and install
REPO_URL = "https://github.com/ziad7amoda/target-ocr-mvp.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os, shutil
if os.path.exists("/content/app-repo"):
    shutil.rmtree("/content/app-repo")
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/app-repo
%cd /content/app-repo
!pip install -q -r requirements.txt -r requirements-gpu.txt
print("installed")


# 4. Start the server and wait for the model

In [ ]:
#@title 4. Start the server and wait for the model
import threading, time, requests, uvicorn
from app.main import app

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()

# Model load plus warm-up takes a while on a cold cache. Poll rather than
# guessing at a sleep duration.
for _ in range(120):
    try:
        h = requests.get("http://127.0.0.1:8000/api/health", timeout=5).json()
        if h.get("loaded"):
            print(h)
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError("Model did not become ready within 10 minutes.")


# 5. Open the public HTTPS tunnel

In [ ]:
#@title 5. Open the public HTTPS tunnel
# Quick tunnel rather than ngrok: no account, no auth token, one less thing
# to fail live.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, threading, time

url = None
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def _watch():
    global url
    for line in proc.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m and not url:
            url = m.group(0)

threading.Thread(target=_watch, daemon=True).start()
for _ in range(60):
    if url:
        break
    time.sleep(1)

print("\n" * 2 + "=" * 72)
print("   OPEN THIS URL:")
print(f"   {url}")
print("=" * 72 + "\n" * 2)
print("Camera capture needs HTTPS, which this tunnel provides.")


# 6. Smoke test before sharing your screen

In [ ]:
#@title 6. Smoke test before sharing your screen
import json, time, requests

IMAGE = "/content/app-repo/eval/samples/synthetic_01.jpg"  #@param {type:"string"}

t0 = time.time()
r = requests.post(
    "http://127.0.0.1:8000/api/extract",
    files={"image": open(IMAGE, "rb")},
    timeout=180,
)
print(f"HTTP {r.status_code} in {time.time() - t0:.1f}s")
body = r.json()
print(json.dumps(body["fields"], indent=2, ensure_ascii=False))
print(f"agreement {body['agreement']}  elapsed_ms {body['elapsed_ms']}")
